# SQL Básico com PySpark

## Antes de começar: tipos de dados e *schema*

Quando criamos uma tabela (seja em um banco relacional tradicional ou no Spark), precisamos dizer, para cada coluna, **que tipo de dado** ela vai guardar. Isso é importante porque o tipo define:

- quanto espaço em memória/disco a coluna ocupa;
- quais operações fazem sentido nela (não dá pra fazer `AVG()` de uma coluna de texto, por exemplo);
- como os valores são comparados e ordenados.

O conjunto de colunas de uma tabela, com seus respectivos nomes e tipos, é chamado de **schema** (esquema). É basicamente a "planta baixa" da tabela. No Spark, podemos sempre visualizar o schema de um DataFrame com `df.printSchema()`.

**Principais tipos de dados que vamos usar (e que já vêm importados na célula abaixo):**

| Tipo Spark | Equivalente em SQL | Uso |
|---|---|---|
| `StringType` | `VARCHAR` / `TEXT` | Texto (nomes, cidades, categorias) |
| `IntegerType` / `LongType` | `INT` / `BIGINT` | Números inteiros |
| `FloatType` / `DoubleType` | `FLOAT` / `DOUBLE` | Números decimais (atenção: podem ter pequenos erros de arredondamento) |
| `DecimalType` | `DECIMAL(p,s)` | Números decimais **exatos** — ideal para dinheiro |
| `BooleanType` | `BOOLEAN` | Verdadeiro/falso |
| `DateType` / `TimestampType` | `DATE` / `TIMESTAMP` | Datas e datas com hora |
| `ArrayType` / `MapType` | — | Coleções (listas e dicionários dentro de uma célula) |

Na aula de hoje vamos usar principalmente `INT`, `VARCHAR` e `FLOAT`, mas é bom já conhecer os outros nomes, porque vão aparecer quando vocês lerem documentação ou mensagens de erro do Spark.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

spark = SparkSession.builder.getOrCreate()

**CRIAR TABELA**

In [ ]:
spark.sql("""
    CREATE TABLE nome_tabela (
        coluna1 TIPO,
        coluna2 TIPO,
        coluna3 TIPO
    )
""")

**ALTERAÇÕES NA TABELA**


*   Renomear
*   Alterar tipo de dado
*   Adicionar nova coluna
*   Deletar

In [ ]:
df = spark.table("nome_tabela")
df = df.withColumnRenamed("coluna_antiga", "coluna_nova")

In [ ]:
df = df.withColumn("coluna", col("coluna").cast("tipo"))

In [ ]:
df.printSchema()

In [ ]:
df = df.withColumn("nova_coluna", lit("valor"))

In [ ]:
spark.sql("DROP TABLE nome_tabela")

**MEXER COM DADOS**

In [ ]:
spark.sql("""
    CREATE TABLE nome_tabela (
        coluna1 TIPO,
        coluna2 TIPO,
        coluna3 TIPO
    )
""")

> **Nota sobre a coluna `nota`:** o ideal, em SQL padrão, seria declarar essa regra já na criação da coluna com uma *constraint*: `nota FLOAT CHECK (nota BETWEEN 0 AND 5)`. Isso funciona em bancos relacionais tradicionais (Postgres, MySQL etc.), mas **o catálogo padrão do Spark usado aqui no Colab não suporta `CHECK` em tabelas comuns** (ele dá erro de "feature not supported") — esse recurso só existe em formatos transacionais como o Delta Lake, que não estamos usando nesta aula. Então, na prática, aplicamos a regra "por fora": validamos os dados depois de inseridos, como na célula abaixo. É o mesmo espaço que estava reservado com a anotação "limitar superiormente a nota".

In [ ]:
spark.sql("SELECT * FROM nome_tabela WHERE coluna < limite_inferior OR coluna > limite_superior").show()

In [ ]:
spark.sql("""
    INSERT INTO nome_tabela VALUES
      (valor1, 'valor2', valor3),
      (valor1, 'valor2', valor3)
""")

**CONSULTAS EM SQL**

In [ ]:
spark.sql("SELECT * FROM nome_tabela").show()

In [ ]:
spark.sql("SELECT coluna1, coluna2 FROM nome_tabela").show()

In [ ]:
spark.sql("""SELECT coluna1, coluna2
FROM nome_tabela
ORDER BY coluna2 DESC
""").show()

In [ ]:
spark.sql("""SELECT MAX(coluna) AS maior_valor
FROM nome_tabela
""").show()

spark.sql("""SELECT MIN(coluna) AS menor_valor
FROM nome_tabela
""").show()

In [ ]:
spark.sql("""SELECT coluna1, coluna2
FROM nome_tabela
ORDER BY coluna2 DESC
LIMIT n
""").show()

In [ ]:
spark.sql("""SELECT coluna_grupo, MAX(coluna_numerica)
FROM nome_tabela
GROUP BY coluna_grupo
""").show()

In [ ]:
spark.sql("""SELECT coluna1 + coluna2 AS resultado, coluna1, coluna2
FROM nome_tabela
""").show()

# Também podem ser usadas -, * e / no lugar de +.

**Operações calculadas e alias (`AS`)**


In [ ]:
spark.sql("""SELECT coluna_grupo, ROUND(AVG(coluna_numerica), 2) AS media
FROM nome_tabela
GROUP BY coluna_grupo
ORDER BY media DESC
LIMIT n
""").show()

> **Sobre o `ROUND()`:** antes, `AVG(salario)` para "padeiro" retornava algo como `10333.333333333334` — um valor com muitas casas decimais, difícil de ler e sem sentido prático para dinheiro. `ROUND(expressão, 2)` arredonda o resultado para 2 casas decimais (centavos). Sempre que uma conta puder gerar dízimas (médias, divisões), vale a pena envolver em `ROUND()`.

In [ ]:
spark.sql("SELECT coluna_grupo, COUNT(coluna_grupo) FROM nome_tabela GROUP BY coluna_grupo").show()

In [ ]:
spark.sql("""SELECT coluna1, coluna2
FROM nome_tabela
WHERE coluna2 > valor
""").show()

In [ ]:
spark.sql("""SELECT ROUND(AVG(coluna_numerica), 2) AS media, coluna_grupo
FROM nome_tabela
WHERE coluna_grupo = 'categoria'
GROUP BY coluna_grupo
""").show()